# VolatilityLab — Feature 2: Implied Volatility Explorer

**Research prototype.** Download current option chains for a selected ticker (or load a saved AAPL snapshot), screen quote quality, **compute implied volatility ourselves from bid/ask mid-quotes**, explore call/put surfaces and an out-of-the-money (OTM) volatility skew, and review data-quality diagnostics.

Live options data are delayed and results change over time; the saved snapshot gives reproducible results. Interpolated surfaces are **exploratory**, not arbitrage-free option-pricing models.

The calculations here mirror `modules/engine.py`, which the Streamlit app uses.

In [1]:
!pip install -q yfinance plotly pandas numpy scipy

zsh:1: command not found: pip


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.interpolate import griddata
from scipy.optimize import brentq
from scipy.stats import norm
from IPython.display import display

# Change these settings and rerun the cells below.
ticker = "AAPL"
DATA_SOURCE = "live"       # "live" (Yahoo Finance) or "snapshot" (saved AAPL quotes)
RISK_FREE_RATE = 0.04      # annual, continuously compounded (decimal)
MAX_EXPIRATIONS = 18

# Saved snapshot (used when DATA_SOURCE = "snapshot" or when live data fail)
SNAPSHOT_FILE = "AAPL_options_snapshot.csv"
SNAPSHOT_SPOT = 344.63     # approximate AAPL price when the snapshot was collected

# Exploratory quote-quality thresholds
MAX_SPREAD = 0.30
STRICT_SPREAD = 0.20
MAX_TRADE_AGE_DAYS = 7
MIN_CONTRACTS_PER_EXPIRY = 5

/Users/hp/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. Method: computing implied volatility ourselves

Yahoo Finance reports an `impliedVolatility` field, but its inputs (price used, rates, dividends, time convention) are undisclosed. A simple consistency check exposes the problem: by **put-call parity**, a European call and put with the same strike and expiry must imply the *same* volatility, yet Yahoo's call and put values often differ by several volatility points (Section 3 measures this). We therefore recompute IV from market quotes.

**Price.** Each contract is valued at its mid-quote, $(\text{bid}+\text{ask})/2$.

**Time to expiry.** $T$ is measured in years (365-day basis) from the quote time to 4:00 pm New York time on the expiration date.

**Forward from put-call parity.** For each expiry, with discount factor $DF = e^{-rT}$,
$$C - P = DF\,(F - K) \quad\Longrightarrow\quad F = K + \frac{C - P}{DF}.$$
We take the median of this estimate over the five strikes nearest spot that have both a call and a put quote. Because the forward is read from the market, expected dividends are already reflected and no dividend-yield input is needed. If an expiry has no call/put pairs, we fall back to $F = S/DF$ (no dividends).

**Black-76 pricing.**
$$C = DF\,[F\,N(d_1) - K\,N(d_2)], \qquad P = DF\,[K\,N(-d_2) - F\,N(-d_1)],$$
$$d_1 = \frac{\ln(F/K) + \tfrac12\sigma^2 T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}.$$

**Solving for $\sigma$.** Brent's method finds the $\sigma \in [0.0001, 5]$ that reproduces the mid-quote. A solution exists only when the price lies inside the no-arbitrage bounds $DF\max(F-K,0) < C < DF\,F$ (calls) or $DF\max(K-F,0) < P < DF\,K$ (puts); otherwise the IV is left missing. We default the risk-free rate to 4.00, which is close to the 3 months T-bill yield of 4.16% on 30 Oct 2026, the day before the snapshot of the AAPL option chain used below.

**Caveat.** Listed US equity options are American. For the OTM contracts used on the main surface the early-exercise premium is small, but deep in-the-money IVs, especially for puts, can be biased upward.

In [3]:
def black76_price(F, K, T, DF, sigma, kind):
    """European option price on a forward F with discount factor DF."""
    sd = sigma * np.sqrt(T)
    d1 = (np.log(F / K) + 0.5 * sd * sd) / sd
    d2 = d1 - sd
    if kind == "Call":
        return DF * (F * norm.cdf(d1) - K * norm.cdf(d2))
    return DF * (K * norm.cdf(-d2) - F * norm.cdf(-d1))


def implied_vol(price, F, K, T, DF, kind, lo=1e-4, hi=5.0):
    """Black-76 implied volatility (decimal); NaN outside the no-arbitrage bounds."""
    values = (price, F, K, T, DF)
    if not all(np.isfinite(v) and v > 0 for v in values):
        return np.nan
    intrinsic = DF * max(F - K, 0.0) if kind == "Call" else DF * max(K - F, 0.0)
    upper = DF * F if kind == "Call" else DF * K
    if not intrinsic < price < upper:
        return np.nan
    try:
        return brentq(lambda s: black76_price(F, K, T, DF, s, kind) - price, lo, hi, xtol=1e-8)
    except ValueError:
        return np.nan


def implied_forward(chain, spot, DF, n_strikes=5):
    """Forward for one expiry from put-call parity; returns (forward, source)."""
    calls = chain[chain["optionType"] == "Call"].groupby("strike")["mid"].median()
    puts = chain[chain["optionType"] == "Put"].groupby("strike")["mid"].median()
    strikes = calls.index.intersection(puts.index)
    if len(strikes) > 0:
        near = strikes[np.argsort(np.abs(strikes.to_numpy() - spot))[:n_strikes]]
        F = float(np.median(near + (calls[near] - puts[near]) / DF))
        if np.isfinite(F) and abs(F / spot - 1) < 0.25:
            return F, "put-call parity"
    return spot / DF, "spot carry (no dividends)"


# Sanity check: price options at known volatilities, then recover them.
F, T, DF = 100.0, 0.5, np.exp(-RISK_FREE_RATE * 0.5)
errors = [abs(implied_vol(black76_price(F, K, T, DF, s, kind), F, K, T, DF, kind) - s)
          for kind in ("Call", "Put") for K in (80, 100, 120) for s in (0.15, 0.30, 0.60)]
print(f"Round-trip check: largest volatility error = {max(errors):.2e}")

Round-trip check: largest volatility error = 7.29e-11


## 2. Retrieve and screen option chains

Both calls and puts are collected across available expirations. If live data cannot be downloaded, the notebook falls back to the saved AAPL snapshot from the repository's `data/` folder (in Colab you will be asked to upload `data/AAPL_options_snapshot.csv`).

Screening keeps contracts with a positive bid, ask above bid, relative spread below 30%, 7–365 days to expiry and moneyness (strike / spot) between 0.70 and 1.30, then requires a computed IV between 5% and 150%. Yahoo's reported IV is retained as `YahooIV_Percent` for comparison only.

In [4]:
def load_live_chain(ticker, max_expirations=MAX_EXPIRATIONS):
    stock = yf.Ticker(ticker)
    history = stock.history(period="5d")
    if history.empty or history["Close"].dropna().empty:
        raise ValueError(f"No stock prices available for {ticker}")
    spot = float(history["Close"].dropna().iloc[-1])
    expirations = stock.options
    if not expirations:
        raise ValueError(f"No option expirations available for {ticker}")
    chunks = []
    for expiration in expirations[:max_expirations]:
        try:
            chain = stock.option_chain(expiration)
        except Exception as exc:
            print(f"Skipped {expiration}: {exc}")
            continue
        for kind, contracts in (("Call", chain.calls), ("Put", chain.puts)):
            if not contracts.empty:
                part = contracts.copy()
                part["optionType"] = kind
                part["expiration"] = expiration
                chunks.append(part)
    if not chunks:
        raise ValueError("No option chains could be downloaded")
    return pd.concat(chunks, ignore_index=True), spot, pd.Timestamp.now(tz="America/New_York")


def find_snapshot():
    for folder in (Path("data"), Path("../data"), Path(".")):
        if (folder / SNAPSHOT_FILE).exists():
            return folder / SNAPSHOT_FILE
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"{SNAPSHOT_FILE} not found; copy it from the repository's data/ folder.")
    print(f"Upload {SNAPSHOT_FILE} from the repository's data/ folder:")
    uploaded = files.upload()
    return Path(next(iter(uploaded)))


def load_snapshot_chain():
    raw = pd.read_csv(find_snapshot())
    as_of = pd.to_datetime(raw["snapshotTimestampUTC"].iloc[0], utc=True)
    return raw, SNAPSHOT_SPOT, as_of


if DATA_SOURCE == "live":
    try:
        raw_options, spot, as_of = load_live_chain(ticker)
        source_label = "Live Yahoo Finance"
    except Exception as exc:
        print(f"Live data unavailable for {ticker} ({exc}). Falling back to the saved AAPL snapshot.")
        DATA_SOURCE = "snapshot"
if DATA_SOURCE == "snapshot":
    raw_options, spot, as_of = load_snapshot_chain()
    ticker, source_label = "AAPL", "Saved snapshot"

as_of = as_of.tz_convert("America/New_York")
print(f"{source_label} · {ticker} · spot ${spot:.2f} · quotes as of {as_of:%Y-%m-%d %H:%M} New York time")
print(f"Raw contracts: {len(raw_options)}")

Live Yahoo Finance · AAPL · spot $344.17 · quotes as of 2026-11-05 19:47 New York time
Raw contracts: 2341


In [5]:
def prepare_chain(raw, spot, as_of, rate=RISK_FREE_RATE):
    d = raw.copy()
    for col in ("strike", "bid", "ask", "impliedVolatility"):
        d[col] = pd.to_numeric(d[col], errors="coerce")

    # Quote-quality screens
    d = d[(d["bid"] > 0) & (d["ask"] > d["bid"])].copy()
    d["mid"] = (d["bid"] + d["ask"]) / 2
    d["relativeSpread"] = (d["ask"] - d["bid"]) / d["mid"]
    d = d[d["relativeSpread"] < MAX_SPREAD].copy()

    # Time to expiry (options stop trading at 4 pm New York time)
    d["expiration"] = pd.to_datetime(d["expiration"])
    expiry_close = (d["expiration"] + pd.Timedelta(hours=16)).dt.tz_localize("America/New_York")
    d["T"] = (expiry_close - as_of).dt.total_seconds() / (365 * 86400)
    d["daysToExpiration"] = (d["expiration"] - as_of.normalize().tz_localize(None)).dt.days
    d = d[d["daysToExpiration"].between(7, 365)].copy()
    d["discountFactor"] = np.exp(-rate * d["T"])

    # One forward per expiry, estimated before the moneyness screen
    forwards = {exp: implied_forward(g, spot, g["discountFactor"].iloc[0])
                for exp, g in d.groupby("expiration")}
    d["forward"] = d["expiration"].map(lambda e: forwards[e][0])
    d["forwardSource"] = d["expiration"].map(lambda e: forwards[e][1])

    d["Moneyness"] = d["strike"] / spot
    d["ForwardMoneyness"] = d["strike"] / d["forward"]
    d = d[d["Moneyness"].between(0.70, 1.30)].copy()

    d["YahooIV_Percent"] = d["impliedVolatility"] * 100
    d["IV_Percent"] = 100 * np.array([
        implied_vol(row.mid, row.forward, row.strike, row.T, row.discountFactor, row.optionType)
        for row in d.itertuples()
    ], dtype=float)
    d = d[d["IV_Percent"].between(5, 150)].copy()
    if d.empty:
        raise ValueError("No contracts passed the quality filters")
    return d.reset_index(drop=True)


options = prepare_chain(raw_options, spot, as_of)
print(f"Screened contracts: {len(options)} | expirations: {options['expiration'].nunique()}")

forward_table = (options.groupby("expiration")
                 .agg(days=("daysToExpiration", "first"), T_years=("T", "first"),
                      forward=("forward", "first"), source=("forwardSource", "first"),
                      contracts=("strike", "size")))
forward_table["forward / spot"] = forward_table["forward"] / spot
display(forward_table.round(4))
display(options[["optionType", "strike", "expiration", "daysToExpiration", "mid",
                 "IV_Percent", "YahooIV_Percent", "relativeSpread"]].head())

Screened contracts: 645 | expirations: 12


,days,T_years,forward,source,contracts,forward / spot
expiration,,,,,,
2026-11-13,8,0.0215,344.7541,put-call parity,76,1.0017
2026-11-20,15,0.0407,344.9750,put-call parity,72,1.0023
2026-11-27,22,0.0598,345.1504,put-call parity,50,1.0028
2026-12-04,29,0.0790,345.7867,put-call parity,40,1.0047
2026-12-11,36,0.0982,346.0399,put-call parity,23,1.0054
2026-12-18,43,0.1174,346.0063,put-call parity,61,1.0053
2027-01-15,71,0.1941,347.2034,put-call parity,67,1.0088
2027-02-19,106,0.2900,348.4319,put-call parity,35,1.0124
2027-03-19,134,0.3666,349.5180,put-call parity,39,1.0155


,optionType,strike,expiration,daysToExpiration,mid,IV_Percent,YahooIV_Percent,relativeSpread
0,Call,245.0,2026-11-13,8,99.675,73.635535,118.310955,0.025583
1,Call,250.0,2026-11-13,8,94.850,97.464937,118.115644,0.031629
2,Call,255.0,2026-11-13,8,89.975,99.168852,115.381282,0.030564
3,Call,260.0,2026-11-13,8,84.775,80.019781,103.516107,0.028900
4,Call,265.0,2026-11-13,8,79.850,80.818733,99.609379,0.028804


## 3. Validation: computed IV vs Yahoo-reported IV

If the inputs are consistent, a call and a put with the same strike and expiry should give nearly the same implied volatility. The table compares the call-minus-put IV gap for Yahoo's values and for ours; the chart shows both for the expiry with the most matched call/put pairs.

In [6]:
pairs = options.pivot_table(index=["expiration", "strike"], columns="optionType",
                            values=["IV_Percent", "YahooIV_Percent"]).dropna()
gap_rows = []
for col, label in (("YahooIV_Percent", "Yahoo-reported"), ("IV_Percent", "Computed (Black-76, mid)")):
    gap = pairs[(col, "Call")] - pairs[(col, "Put")]
    gap_rows.append({"IV source": label, "Matched pairs": len(gap),
                     "Median call − put gap (vol pts)": gap.median(),
                     "Median |gap|": gap.abs().median(), "90th percentile |gap|": gap.abs().quantile(0.9)})
display(pd.DataFrame(gap_rows).round(2))

compare_expiry = pairs.groupby(level="expiration").size().idxmax()
smile = options[(options["expiration"] == compare_expiry) &
                options["Moneyness"].between(0.80, 1.20)].sort_values("strike")
fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=("Yahoo-reported IV", "Computed IV (mid-quote, Black-76)"))
for col_idx, column in ((1, "YahooIV_Percent"), (2, "IV_Percent")):
    for kind, color in (("Call", "#2a9d8f"), ("Put", "#7b2cbf")):
        sub = smile[smile["optionType"] == kind]
        fig.add_trace(go.Scatter(x=sub["Moneyness"], y=sub[column], mode="lines+markers",
                                 name=kind, line_color=color, showlegend=(col_idx == 1)),
                      row=1, col=col_idx)
    fig.add_vline(x=1, line_dash="dash", line_color="gray", row=1, col=col_idx)
fig.update_xaxes(title_text="Strike / spot")
fig.update_yaxes(title_text="IV (%)", row=1, col=1)
fig.update_layout(title=f"{ticker} — calls vs puts, expiry {compare_expiry.date()}", height=450)
fig.show()

,IV source,Matched pairs,Median call − put gap (vol pts),Median |gap|,90th percentile |gap|
0,Yahoo-reported,265,8.01,8.11,12.89
1,"Computed (Black-76, mid)",265,0.18,0.77,2.99


## 4. Explore separate call and put volatility surfaces

These surfaces use our computed IV and linear interpolation only inside the observed points' convex hull; blank areas are intentional. Individual call and put surfaces can be selected from the dropdown. Because these include in-the-money contracts, early exercise (see Section 1) can lift deep in-the-money put IVs.

In [7]:
def make_surface(df, x_col="strike", min_points=10):
    points = df.groupby([x_col, "daysToExpiration"], as_index=False)["IV_Percent"].median()
    if (len(points) < min_points or points[x_col].nunique() < 2
            or points["daysToExpiration"].nunique() < 2):
        return None
    X, Y = np.meshgrid(
        np.linspace(points[x_col].min(), points[x_col].max(), 60),
        np.linspace(points["daysToExpiration"].min(), points["daysToExpiration"].max(), 60)
    )
    try:
        Z = griddata((points[x_col], points["daysToExpiration"]),
                     points["IV_Percent"], (X, Y), method="linear")
    except Exception:
        return None
    if np.isnan(Z).all():
        return None
    return X, Y, Z, points

fig = go.Figure()
labels = []
for kind in ("Call", "Put"):
    result = make_surface(options.loc[options["optionType"] == kind])
    if result is None:
        print(f"Not enough points for {kind} surface")
        continue
    X, Y, Z, _ = result
    fig.add_trace(go.Surface(x=X, y=Y, z=Z, name=kind,
                             visible=(len(labels) == 0), colorscale="Viridis"))
    labels.append(kind)
if not labels:
    print("No call/put surface can be interpolated with available quotes.")
else:
    buttons = [dict(label=kind + "s", method="update",
                    args=[{"visible": [j == i for j in range(len(labels))]}])
               for i, kind in enumerate(labels)]
    fig.update_layout(title=f"{ticker} — Call / Put IV Surface (computed IV)",
        scene=dict(xaxis_title="Strike ($)", yaxis_title="Days to expiration",
                   zaxis_title="Implied volatility (%)"),
        updatemenus=[dict(buttons=buttons, direction="down", x=0.05, y=1.10)],
        height=700)
    fig.show()

## 5. OTM skew and volatility smile

We use puts for strikes below spot (moneyness < 1) and calls for strikes above spot (moneyness > 1) because these out-of-the-money options have no intrinsic value, so their prices consist entirely of time value and therefore reflect volatility directly; they are also more liquid, more tightly quoted, and less affected by early exercise than their in-the-money counterparts. We focus on 85%–115% moneyness and 14–365 days. The 2D chart selects the expiration with the most retained contracts; Yahoo-reported values are shown as open markers for comparison. 

In [9]:
clean_surface = options.loc[
    (((options["optionType"] == "Call") & (options["strike"] >= spot)) |
     ((options["optionType"] == "Put") & (options["strike"] <= spot))) &
    options["Moneyness"].between(0.85, 1.15) &
    options["daysToExpiration"].between(14, 365)
].copy()
print("OTM contracts:", len(clean_surface), "| expirations:", clean_surface["expiration"].nunique())
if clean_surface.empty:
    print("No OTM contracts passed the filters.")
else:
    selected_expiration = clean_surface.groupby("expiration").size().idxmax()
    smile = clean_surface.loc[clean_surface["expiration"] == selected_expiration].sort_values("Moneyness")
    fig_smile = px.scatter(smile, x="Moneyness", y="IV_Percent", color="optionType",
                           hover_data=["strike", "bid", "ask", "YahooIV_Percent"],
                           title=f"{ticker} — OTM IV Skew ({selected_expiration.date()})")
    fig_smile.update_traces(mode="lines+markers")
    fig_smile.add_trace(go.Scatter(x=smile["Moneyness"], y=smile["YahooIV_Percent"], mode="markers",
                                   name="Yahoo-reported", marker=dict(symbol="circle-open", size=7, color="gray")))
    fig_smile.add_vline(x=1, line_dash="dash", line_color="gray")
    fig_smile.update_layout(xaxis_title="Strike / Spot", yaxis_title="IV (%)")
    fig_smile.show()

OTM contracts: 173 | expirations: 11


## 6. Quote-quality diagnostics and final quality-filtered surface

`lastTradeDate` measures the age of the last **trade**, not the quote. Trade age is measured from the quote time (the snapshot time when using saved data). Strict screening requires a relative spread at most 20% and a last trade no older than seven days. Sparse expirations are omitted. Observed points appear on the final surface so interpolation gaps remain visible.

In [16]:
quality_data = clean_surface.copy()
quality_data["lastTradeDate"] = pd.to_datetime(quality_data["lastTradeDate"], utc=True, errors="coerce")
quality_data["tradeAgeDays"] = (as_of.tz_convert("UTC") - quality_data["lastTradeDate"]).dt.total_seconds() / 86400
quality_data["staleTrade"] = quality_data["tradeAgeDays"] > MAX_TRADE_AGE_DAYS
quality_data["wideSpread"] = quality_data["relativeSpread"] > STRICT_SPREAD
summary = pd.DataFrame({
    "Metric": ["OTM contracts", "Expirations", "Median trade age (days)",
               "Trades older than 7 days (%)", "Spreads above 20% (%)", "Median computed IV (%)",
               "Median |computed − Yahoo| IV (vol pts)"],
    "Value": [len(quality_data), quality_data["expiration"].nunique(),
              quality_data["tradeAgeDays"].median(), quality_data["staleTrade"].mean() * 100,
              quality_data["wideSpread"].mean() * 100, quality_data["IV_Percent"].median(),
              (quality_data["IV_Percent"] - quality_data["YahooIV_Percent"]).abs().median()]
})
display(summary.round(2))

high_quality_surface = quality_data.loc[
    (quality_data["relativeSpread"] <= STRICT_SPREAD) &
    quality_data["tradeAgeDays"].between(0, MAX_TRADE_AGE_DAYS)
].copy()
counts = high_quality_surface.groupby("expiration").size()
valid_expirations = counts[counts >= MIN_CONTRACTS_PER_EXPIRY].index
final_data = high_quality_surface.loc[high_quality_surface["expiration"].isin(valid_expirations)]
print("Strict contracts:", len(high_quality_surface), "| final contracts:", len(final_data))
display(counts.rename("Contracts").reset_index())

result = make_surface(final_data, x_col="Moneyness")
if result is None:
    print("Not enough high-quality observations for a reliable interpolated surface. Review the observed data instead.")
else:
    X, Y, Z, observed = result
    fig_final = go.Figure()
    fig_final.add_trace(go.Surface(x=X, y=Y, z=Z, colorscale="Viridis", opacity=0.80,
                                   name="Interpolated IV"))
    fig_final.add_trace(go.Scatter3d(x=observed["Moneyness"], y=observed["daysToExpiration"],
                                     z=observed["IV_Percent"], mode="markers",
                                     marker=dict(size=4, color="red"), name="Observed contracts"))
    fig_final.update_layout(title=f"{ticker} — Quality-Filtered OTM IV Surface (computed IV)",
        scene=dict(xaxis_title="Moneyness (strike / spot)", yaxis_title="Days to expiration",
                   zaxis_title="Implied volatility (%)"), height=700)
    fig_final.show()

,Metric,Value
0,OTM contracts,173.00
1,Expirations,11.00
2,Median trade age (days),0.15
3,Trades older than 7 days (%),0.00
4,Spreads above 20% (%),7.51
5,Median computed IV (%),28.38
6,Median |computed − Yahoo| IV (vol pts),1.96


Strict contracts: 160 | final contracts: 160


,expiration,Contracts
0,2026-11-20,23
1,2026-11-27,11
2,2026-12-04,8
3,2026-12-11,5
4,2026-12-18,20
5,2027-01-15,21
6,2027-02-19,10
7,2027-03-19,10
8,2027-04-16,21
9,2027-06-17,10


## Interpretation and limitations

- **Implied volatility is computed, not downloaded.** IVs come from bid/ask mid-quotes using Black-76 with a forward implied from put-call parity for each expiry and the chosen risk-free rate. Section 3 shows that this makes call and put IVs consistent, which Yahoo's reported values are not.
- The **3D surface** illustrates implied volatility variation across strike and expiration; the **2D skew** isolates one expiration.
- Listed equity options are **American**; treating them as European slightly overstates IV where early exercise has value (mainly deep in-the-money puts). The OTM surface is little affected.
- Mid-quotes are not executable prices, and wide spreads make IV uncertain. The dataset is a single snapshot, not historical options quotes; Yahoo Finance values may be delayed, stale, missing, or inconsistent.
- Screening and linear interpolation are exploratory. They do not ensure no-arbitrage consistency across strikes and maturities, executable prices, or reliable Greeks.
- `lastTradeDate` is **not** a timestamp for bid/ask quotes. Contracts and expiration coverage vary by ticker and date.
- For a competition demonstration, record the data source, ticker, quote time, risk-free rate, retained contract counts, and assumptions; do not present the interpolated surface as a calibrated option-pricing model.